# 🏦 German Mark — Monetary History EDA (1871–1948)
## Goldmark · Hyperinflation 1923 · Rentenmark · Reichsmark · 77 Years

**Dataset:** German Mark Complete Monetary History  
**Author:** Sergey Nefedov | [github.com/Sergpreneur](https://github.com/Sergpreneur)

---

### What this notebook covers
1. 📊 Overview — 77 years of monetary history across 4 eras
2. 💱 Exchange rate — from 4.20 to 4.2 trillion Mark/USD
3. 🔥 Hyperinflation anatomy — 1921–1923 in detail
4. 🍞 The human cost — bread, coal and rent prices
5. 🏦 Monetary mechanics — money supply, gold coverage, velocity
6. 📉 Economic collapse — GDP, unemployment, the Great Depression
7. 📰 Event study — geopolitical shocks and market impact
8. 🔬 Quantity Theory of Money — does MV = PQ hold?

> **Key numbers:**  
> January 1913: 1 USD = **4.20** Mark  
> November 1923: 1 USD = **4,200,000,000,000** Mark  
> That is a **1 trillion× depreciation** in 10 years.  
> A loaf of bread: **0.29 Mark** in 1913 → **~200 billion Mark** in November 1923.


## 0. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 130, 'axes.facecolor': '#0d1117', 'figure.facecolor': '#0d1117',
    'axes.edgecolor': '#30363d', 'axes.labelcolor': '#c9d1d9',
    'xtick.color': '#8b949e', 'ytick.color': '#8b949e', 'text.color': '#c9d1d9',
    'grid.color': '#21262d', 'grid.alpha': 0.5,
    'axes.spines.top': False, 'axes.spines.right': False,
})
GOLD='#e8a020'; RED='#f85149'; BLUE='#388bfd'; GREEN='#3fb950'
AMBER='#f7931a'; PURPLE='#9945ff'; TEAL='#39d353'; GRAY='#8b949e'

ERA_COLORS = {
    'Goldmark': GOLD, 'Papiermark': RED,
    'Rentenmark': GREEN, 'Reichsmark': BLUE
}
ERA_ORDER = ['Goldmark','Papiermark','Rentenmark','Reichsmark']

PATH = '/kaggle/input/datasets/sergionefedov/german-mark-monetary-history-1871-1948/'

fx     = pd.read_csv(PATH + 'exchange_rates.csv')
inf    = pd.read_csv(PATH + 'inflation_data.csv')
ms     = pd.read_csv(PATH + 'monetary_supply.csv')
econ   = pd.read_csv(PATH + 'economic_indicators.csv')
events = pd.read_csv(PATH + 'historical_events.csv')

# Parse dates
for df in [fx, inf, ms]:
    df['date_dt'] = pd.to_datetime(df['date'] + '-01')

print(f"Exchange rates: {len(fx):,} monthly obs | {fx['year'].min()}–{fx['year'].max()}")
print(f"Inflation:      {len(inf):,} monthly obs")
print(f"Monetary:       {len(ms):,}  monthly obs")
print(f"Economic:       {len(econ):,}  annual obs")
print(f"Events:         {len(events):,}  historical events")
print()
print("Currency eras:")
print(fx.groupby('era')[['year']].agg(['min','max']).to_string())
print()
goldmark_rate = fx[fx['era']=='Goldmark']['mark_per_usd'].mean()
hyper_peak    = fx['mark_per_usd'].max()
print(f"Goldmark avg rate:      {goldmark_rate:.2f} Mark/USD")
print(f"Hyperinflation peak:    {hyper_peak:,.0f} Mark/USD")
print(f"Depreciation factor:    {hyper_peak/goldmark_rate:,.0f}×")


---
## 1. 📊 Overview — 77 Years of Monetary History

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Panel 1: Full exchange rate history (log scale)
ax = axes[0,0]
for era, color in ERA_COLORS.items():
    sub = fx[fx['era']==era]
    ax.plot(sub['date_dt'], sub['log10_mark_per_usd'],
            color=color, linewidth=1.5, label=era)
# Shade eras
era_spans = [
    ('Goldmark',  '1871-01-01','1914-07-01', GOLD),
    ('Papiermark','1914-08-01','1923-11-01', RED),
    ('Rentenmark','1923-11-01','1924-08-01', GREEN),
    ('Reichsmark','1924-09-01','1948-06-01', BLUE),
]
for _, s, e, col in era_spans:
    import matplotlib.patches as mpatches
    ax.axvspan(pd.Timestamp(s), pd.Timestamp(e), alpha=0.06, color=col)
ax.axhline(0, color=GRAY, linewidth=0.7, linestyle='--', alpha=0.5)
ax.set_title('Mark/USD Exchange Rate 1871–1948 (log₁₀ scale)', fontsize=11)
ax.set_ylabel('log₁₀(Mark per USD)')
ax.set_yticks(range(-1, 14, 2))
ax.set_yticklabels(['0.1','10','1K','100K','10M','1B','100B','10T'])
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# Panel 2: CPI history (log scale)
ax = axes[0,1]
for era, color in ERA_COLORS.items():
    sub = inf[inf['era']==era]
    ax.plot(sub['date_dt'], sub['log10_cpi'],
            color=color, linewidth=1.5, label=era)
for _, s, e, col in era_spans:
    ax.axvspan(pd.Timestamp(s), pd.Timestamp(e), alpha=0.06, color=col)
ax.axhline(2, color=GRAY, linewidth=0.7, linestyle='--', alpha=0.5, label='CPI=100 (baseline)')
ax.set_title('Consumer Price Index 1871–1948 (log₁₀, 1913=100)', fontsize=11)
ax.set_ylabel('log₁₀(CPI)')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# Panel 3: Key stats by era
ax = axes[1,0]
era_stats = fx.groupby('era', observed=True).agg(
    avg_rate=('mark_per_usd','mean'),
    n_months=('date','count')
).reindex(ERA_ORDER)
era_stats['log_avg'] = np.log10(era_stats['avg_rate'].clip(0.01))
ax.bar(ERA_ORDER, era_stats['n_months'],
       color=[ERA_COLORS[e] for e in ERA_ORDER], alpha=0.85)
ax.set_title('Duration of Each Currency Era (months)', fontsize=11)
ax.set_ylabel('Months'); ax.grid(True, alpha=0.3, axis='y')
for i,(era,row) in enumerate(era_stats.iterrows()):
    ax.text(i, row['n_months']+3, f"{int(row['n_months'])}m",
            ha='center', fontsize=9)

# Panel 4: GDP and unemployment
ax = axes[1,1]
ax2 = ax.twinx()
ax.fill_between(econ['year'], econ['gdp_index_1913_100'], alpha=0.3, color=BLUE)
ax.plot(econ['year'], econ['gdp_index_1913_100'], color=BLUE, linewidth=2, label='GDP (1913=100)')
ax2.plot(econ['year'], econ['unemployment_rate_pct'], color=RED, linewidth=2,
         linestyle='--', label='Unemployment %')
ax.set_title('Germany: GDP Index & Unemployment 1871–1948', fontsize=11)
ax.set_ylabel('GDP Index (1913=100)', color=BLUE)
ax2.set_ylabel('Unemployment (%)', color=RED)
ax.axvspan(1914,1918, alpha=0.08, color=RED, label='WWI')
ax.axvspan(1939,1945, alpha=0.08, color=AMBER, label='WWII')
lines1,labels1 = ax.get_legend_handles_labels()
lines2,labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1+lines2, labels1+labels2, fontsize=7, ncol=2)
ax.grid(True, alpha=0.3)

plt.suptitle('German Mark — 77-Year Monetary History Overview', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('overview.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()


---
## 2. 💱 Exchange Rate — From Stability to Collapse

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Panel 1: Goldmark era — stable gold standard
ax = axes[0,0]
goldmark = fx[fx['era']=='Goldmark']
ax.fill_between(goldmark['date_dt'], goldmark['mark_per_usd'],
                alpha=0.3, color=GOLD)
ax.plot(goldmark['date_dt'], goldmark['mark_per_usd'], color=GOLD, linewidth=1.5)
ax.axhline(4.20, color=GRAY, linewidth=1, linestyle='--', label='Gold parity 4.20')
ax.set_title('Goldmark Era: Stability (1871–1914)', fontsize=11)
ax.set_ylabel('Mark per USD'); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
std_gold = goldmark['mark_per_usd'].std()
ax.text(0.05, 0.88, f'σ = {std_gold:.3f} (gold standard stability)',
        transform=ax.transAxes, fontsize=8,
        bbox=dict(boxstyle='round', facecolor='#21262d', alpha=0.8))

# Panel 2: WWI collapse (1914–1923)
ax = axes[0,1]
ww1 = fx[(fx['year'] >= 1914) & (fx['year'] <= 1923)]
ax.semilogy(ww1['date_dt'], ww1['mark_per_usd'], color=RED, linewidth=1.5)
ax.fill_between(ww1['date_dt'], ww1['mark_per_usd'], alpha=0.2, color=RED)
# Mark key dates
for date_s, label in [('1914-08','WWI start'),('1918-11','Armistice'),
                       ('1919-06','Versailles'),('1923-01','Ruhr')]:
    sub = fx[fx['date']==date_s]
    if not sub.empty:
        ax.axvline(sub['date_dt'].iloc[0], color=AMBER, linewidth=0.8, linestyle=':')
        ax.text(sub['date_dt'].iloc[0], sub['mark_per_usd'].iloc[0]*3,
                label, fontsize=7, color=AMBER, rotation=35, ha='left')
ax.set_title('WWI & Hyperinflation: Collapse (1914–1923, log)', fontsize=11)
ax.set_ylabel('Mark per USD (log scale)'); ax.grid(True, alpha=0.3)

# Panel 3: Hyperinflation in detail (1921–1923 only)
ax = axes[0,2]
hyper = fx[fx['is_hyperinflation']==1]
ax.semilogy(hyper['date_dt'], hyper['mark_per_usd'], color=RED, linewidth=2.5,
            marker='o', markersize=4)
ax.fill_between(hyper['date_dt'], hyper['mark_per_usd'], alpha=0.25, color=RED)
ax.set_title('Terminal Hyperinflation Detail (1921–1923)', fontsize=11)
ax.set_ylabel('Mark per USD (log scale)'); ax.grid(True, alpha=0.3)
# Annotate peak
peak_idx = hyper['mark_per_usd'].idxmax()
ax.text(0.02,0.88,f'Peak rate: see Nov 1923',transform=ax.transAxes,fontsize=9,color=RED,bbox=dict(boxstyle='round',facecolor='#21262d',alpha=0.8))
            #annotation removed




# Panel 4: Rentenmark miracle (1923-1924)
ax = axes[1,0]
renten_window = fx[(fx['year'] >= 1923) & (fx['year'] <= 1925)]
ax.semilogy(renten_window['date_dt'], renten_window['mark_per_usd'],
            color=GREEN, linewidth=2, marker='o', markersize=5)
ax.axvline(pd.Timestamp('1923-11-15'), color=GREEN, linewidth=2,
           linestyle='--', label='Rentenmark introduced')
ax.axhline(4.20, color=GOLD, linewidth=1.5, linestyle=':', label='Old parity 4.20')
ax.set_title('The Rentenmark Miracle: Overnight Stabilization', fontsize=11)
ax.set_ylabel('Mark per USD (log)'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# Panel 5: Reichsmark stability (1924–1945)
ax = axes[1,1]
reich = fx[fx['era']=='Reichsmark']
ax.fill_between(reich['date_dt'], reich['mark_per_usd'], alpha=0.2, color=BLUE)
ax.plot(reich['date_dt'], reich['mark_per_usd'], color=BLUE, linewidth=1.5)
ax.axvline(pd.Timestamp('1933-01-30'), color=RED, linewidth=1,
           linestyle='--', label='Hitler chancellor')
ax.axvline(pd.Timestamp('1939-09-01'), color=AMBER, linewidth=1,
           linestyle='--', label='WWII starts')
ax.set_title('Reichsmark Era: Controlled Rate (1924–1948)', fontsize=11)
ax.set_ylabel('Mark per USD')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# Panel 6: Black market premium in Nazi era
ax = axes[1,2]
nazi_era = fx[(fx['year'] >= 1933) & (fx['year'] <= 1948)]
ax.fill_between(nazi_era['date_dt'], nazi_era['parallel_market_premium_pct'],
                alpha=0.5, color=RED)
ax.plot(nazi_era['date_dt'], nazi_era['parallel_market_premium_pct'],
        color=RED, linewidth=1.5)
ax.axvline(pd.Timestamp('1939-09-01'), color=AMBER, linewidth=1,
           linestyle='--', label='WWII starts')
ax.axvline(pd.Timestamp('1945-05-08'), color=GREEN, linewidth=1,
           linestyle='--', label='Germany surrenders')
ax.set_title('Black Market Premium vs Official Rate (%)', fontsize=11)
ax.set_ylabel('Premium (%)'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.suptitle('Exchange Rate Analysis Across All Eras', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('exchange_rates.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()

# Key stats
pre_war  = fx[fx['date']=='1914-07']['mark_per_usd'].iloc[0]
post_war = fx[fx['date']=='1918-12']['mark_per_usd'].iloc[0]
peak     = fx['mark_per_usd'].max()
print(f"Pre-WWI rate:    {pre_war:.2f} Mark/USD")
print(f"Post-WWI rate:   {post_war:.2f} Mark/USD (+{post_war/pre_war:.1f}x)")
print(f"Hyperinfl. peak: {peak:,.0f} Mark/USD ({peak/pre_war:.2e}x pre-war)")


---
## 3. 🔥 Hyperinflation Anatomy — Month by Month

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

hyper_inf = inf[(inf['year'] >= 1920) & (inf['year'] <= 1924)].copy()
hyper_fx  = fx[(fx['year']  >= 1920) & (fx['year']  <= 1924)].copy()

# Panel 1: Month-over-month inflation rate
ax = axes[0,0]
mom = hyper_inf[hyper_inf['mom_inflation_pct'].notna()].copy()
ax.bar(mom['date_dt'], mom['mom_inflation_pct'].clip(0, 100000),
       color=[RED if v > 50 else AMBER for v in mom['mom_inflation_pct']], alpha=0.85)
ax.set_title('Month-over-Month Inflation Rate (%)', fontsize=11)
ax.set_ylabel('MoM Inflation (%)'); ax.grid(True, alpha=0.3, axis='y')
ax.set_yscale('symlog', linthresh=10)
peak_mom = mom.loc[mom['mom_inflation_pct'].idxmax()]
ax.text(0.05, 0.88, f"Peak month: {peak_mom['date']} (+{peak_mom['mom_inflation_pct']:,.0f}%)",
        transform=ax.transAxes, fontsize=8, color=RED,
        bbox=dict(boxstyle='round', facecolor='#21262d', alpha=0.8))

# Panel 2: Exchange rate acceleration
ax = axes[0,1]
log_diff = np.diff(hyper_fx['log10_mark_per_usd'].values)
ax.bar(hyper_fx['date_dt'].iloc[1:], log_diff,
       color=[RED if v > 0 else GREEN for v in log_diff], alpha=0.85)
ax.axhline(0, color=GRAY, linewidth=0.8)
ax.set_title('Monthly Change in log₁₀(Mark/USD) — Rate of Acceleration', fontsize=11)
ax.set_ylabel('Δ log₁₀ (Mark/USD)'); ax.grid(True, alpha=0.3, axis='y')
ax.axvline(pd.Timestamp('1923-01-01'), color=AMBER, linewidth=1.5,
           linestyle='--', label='Ruhr occupation')
ax.axvline(pd.Timestamp('1923-11-15'), color=GREEN, linewidth=1.5,
           linestyle='--', label='Rentenmark')
ax.legend(fontsize=8)

# Panel 3: Comparison — exchange rate vs money supply
ax = axes[1,0]
hyper_ms = ms[(ms['year'] >= 1920) & (ms['year'] <= 1924)]
ax.plot(hyper_fx['date_dt'], hyper_fx['log10_mark_per_usd'],
        color=RED, linewidth=2, label='log₁₀(Mark/USD)')
ax.plot(hyper_ms['date_dt'], hyper_ms['log10_money_supply'] - 9,
        color=AMBER, linewidth=2, linestyle='--', label='log₁₀(Money supply) - 9')
ax.plot(hyper_inf[hyper_inf['year'].between(1920,1924)]['date_dt'],
        hyper_inf[hyper_inf['year'].between(1920,1924)]['log10_cpi'] - 2,
        color=BLUE, linewidth=2, linestyle=':', label='log₁₀(CPI) - 2')
ax.set_title('Hyperinflation: FX Rate vs Money Supply vs CPI (log scale)', fontsize=11)
ax.set_ylabel('log₁₀ (shifted for visibility)')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
r,_ = stats.pearsonr(
    hyper_fx['log10_mark_per_usd'].values,
    hyper_ms['log10_money_supply'].values
)
ax.text(0.05, 0.88, f'Correlation: r = {r:.3f}',
        transform=ax.transAxes, fontsize=9,
        bbox=dict(boxstyle='round', facecolor='#21262d', alpha=0.8))

# Panel 4: Real wage collapse
ax = axes[1,1]
wage = inf[(inf['year'] >= 1919) & (inf['year'] <= 1925)]
ax.fill_between(wage['date_dt'], wage['real_wage_index'], alpha=0.3, color=BLUE)
ax.plot(wage['date_dt'], wage['real_wage_index'], color=BLUE, linewidth=2)
ax.axhline(100, color=GOLD, linewidth=1.5, linestyle='--', label='1913 baseline')
ax.axvline(pd.Timestamp('1923-11-15'), color=GREEN, linewidth=1.5,
           linestyle='--', label='Rentenmark')
ax.set_title('Real Wage Index During Hyperinflation (1913=100)', fontsize=11)
ax.set_ylabel('Real Wage Index')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
trough = wage['real_wage_index'].min()
trough_date = wage.loc[wage['real_wage_index'].idxmin(), 'date']
ax.text(0.05, 0.12, f'Trough: {trough:.1f} ({trough_date})',
        transform=ax.transAxes, fontsize=9, color=RED,
        bbox=dict(boxstyle='round', facecolor='#21262d', alpha=0.8))

plt.suptitle('Hyperinflation Anatomy 1920–1924', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('hyperinflation.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()

print(f"Peak MoM inflation: {mom['mom_inflation_pct'].max():,.0f}% ({peak_mom['date']})")
print(f"FX vs money supply correlation (log-log): r = {r:.3f}")
print(f"Real wage trough: {trough:.1f} of 1913 baseline ({trough_date})")


---
## 4. 🍞 The Human Cost — Prices in Everyday Life

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Focus: full period but with era shading
def add_era_bands(ax):
    for _, s, e, col in [
        ('Goldmark','1871-01-01','1914-07-01', GOLD),
        ('Papiermark','1914-08-01','1923-11-01', RED),
        ('Rentenmark','1923-11-01','1924-08-01', GREEN),
        ('Reichsmark','1924-09-01','1948-06-01', BLUE)]:
        ax.axvspan(pd.Timestamp(s), pd.Timestamp(e), alpha=0.06, color=col)

# Panel 1: Bread price (log scale)
ax = axes[0,0]
ax.semilogy(inf['date_dt'], inf['rye_bread_mark_kg'].clip(1e-6),
            color=AMBER, linewidth=1.5)
add_era_bands(ax)
ax.set_title('Rye Bread Price (Mark/kg, log scale)', fontsize=11)
ax.set_ylabel('Mark per kg (log)')
ax.axhline(0.29, color=GRAY, linewidth=0.7, linestyle='--', label='1913 baseline 0.29')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# Panel 2: Coal price (essential for heating)
ax = axes[0,1]
ax.semilogy(inf['date_dt'], inf['coal_mark_100kg'].clip(1e-6),
            color=GRAY, linewidth=1.5)
add_era_bands(ax)
ax.set_title('Coal Price (Mark/100kg, log scale)', fontsize=11)
ax.set_ylabel('Mark per 100kg (log)'); ax.grid(True, alpha=0.3)

# Panel 3: Price basket during hyperinflation only
ax = axes[1,0]
h_inf = inf[(inf['year'] >= 1921) & (inf['year'] <= 1924)].copy()
ax.semilogy(h_inf['date_dt'], h_inf['rye_bread_mark_kg'], color=AMBER,
            linewidth=2, label='Bread (Mark/kg)')
ax.semilogy(h_inf['date_dt'], h_inf['beef_mark_kg'], color=RED,
            linewidth=2, label='Beef (Mark/kg)')
ax.semilogy(h_inf['date_dt'], h_inf['butter_mark_kg'], color=GOLD,
            linewidth=2, label='Butter (Mark/kg)')
ax.semilogy(h_inf['date_dt'], h_inf['tram_ticket_mark'], color=BLUE,
            linewidth=2, label='Tram ticket')
ax.set_title('Consumer Price Basket: Hyperinflation Detail (log)', fontsize=11)
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# Panel 4: Price level in three reference points
ax = axes[1,1]
ref_dates = ['1913-01','1919-01','1921-01','1922-01','1923-01','1923-07','1923-11','1924-01','1929-01']
products  = ['rye_bread_mark_kg','beef_mark_kg','coal_mark_100kg','monthly_rent_mark']
labels_p  = ['Bread/kg','Beef/kg','Coal/100kg','Monthly rent']
colors_p  = [AMBER,RED,GRAY,BLUE]

ref_data = inf[inf['date'].isin(ref_dates)].set_index('date')
norm = ref_data.loc['1913-01', products].values

x = np.arange(len(ref_dates))
w = 0.18
for j,(prod,label,col) in enumerate(zip(products,labels_p,colors_p)):
    vals = [ref_data.loc[d, prod]/norm[j] if d in ref_data.index else np.nan
            for d in ref_dates]
    ax.bar(x + j*w, vals, w, label=label, color=col, alpha=0.85)

ax.set_xticks(x + w*1.5)
ax.set_xticklabels(ref_dates, rotation=45, ha='right', fontsize=7)
ax.set_yscale('symlog', linthresh=100)
ax.set_title('Price Index vs 1913 Baseline (log scale)', fontsize=11)
ax.set_ylabel('× baseline (1913=1)')
ax.legend(fontsize=7, ncol=2); ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Consumer Prices: The Human Dimension of Hyperinflation', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('consumer_prices.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()

bread_1913 = inf[inf['date']=='1913-01']['rye_bread_mark_kg'].iloc[0]
bread_1923 = inf[inf['date']=='1923-11']['rye_bread_mark_kg'].iloc[0]
print(f"Bread 1913: {bread_1913:.3f} Mark/kg")
print(f"Bread Nov 1923: {bread_1923:,.0f} Mark/kg")
print(f"Increase: {bread_1923/bread_1913:,.0f}×")


---
## 5. 🏦 Monetary Mechanics — Money Supply & Gold Coverage

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Panel 1: Money supply over full period (log)
ax = axes[0,0]
for era, color in ERA_COLORS.items():
    sub = ms[ms['era']==era]
    ax.semilogy(sub['date_dt'], sub['money_supply_mark'], color=color, linewidth=2, label=era)
ax.set_title('Money in Circulation (Mark, log scale)', fontsize=11)
ax.set_ylabel('Mark (log scale)'); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# Panel 2: Gold coverage ratio
ax = axes[0,1]
ax.fill_between(ms['date_dt'], ms['gold_coverage_pct'].clip(0,100),
                alpha=0.3, color=GOLD)
ax.plot(ms['date_dt'], ms['gold_coverage_pct'].clip(0,100), color=GOLD, linewidth=1.5)
ax.axhline(100, color=GRAY, linewidth=0.8, linestyle='--', label='Full gold coverage')
ax.axhline(40, color=AMBER, linewidth=0.8, linestyle=':', label='Minimum required (40%)')
ax.set_title('Gold Reserve Coverage Ratio (%)', fontsize=11)
ax.set_ylabel('%'); ax.set_ylim(0, 110)
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# Panel 3: Reichsbank discount rate
ax = axes[1,0]
ax.fill_between(ms['date_dt'], ms['reichsbank_discount_rate'],
                alpha=0.3, color=RED,
                where=ms['reichsbank_discount_rate'] > 10)
ax.fill_between(ms['date_dt'], ms['reichsbank_discount_rate'],
                alpha=0.3, color=GREEN,
                where=ms['reichsbank_discount_rate'] <= 10)
ax.plot(ms['date_dt'], ms['reichsbank_discount_rate'], color='#c9d1d9', linewidth=1)
ax.set_title('Reichsbank Discount Rate (%)', fontsize=11)
ax.set_ylabel('%'); ax.set_yscale('symlog', linthresh=20)
ax.grid(True, alpha=0.3)
peak_rate_idx = ms['reichsbank_discount_rate'].idxmax()
ax.text(0.05, 0.88,
        f"Peak: {ms.loc[peak_rate_idx,'reichsbank_discount_rate']:.0f}% ({ms.loc[peak_rate_idx,'date']})",
        transform=ax.transAxes, fontsize=8, color=RED,
        bbox=dict(boxstyle='round', facecolor='#21262d', alpha=0.8))

# Panel 4: MEFO bills — secret Nazi rearmament
ax = axes[1,1]
nazi_ms = ms[(ms['year'] >= 1933) & (ms['year'] <= 1942)]
ax.fill_between(nazi_ms['date_dt'], nazi_ms['mefo_bills_million_rm'],
                alpha=0.5, color=RED)
ax.plot(nazi_ms['date_dt'], nazi_ms['mefo_bills_million_rm'],
        color=RED, linewidth=2)
ax.set_title('MEFO Bills vs Unemployment', fontsize=11)
ax.set_ylabel('Million Reichsmark'); ax.grid(True, alpha=0.3)







plt.suptitle('Monetary Mechanics: Money Supply, Gold Reserves & Central Bank Policy', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('monetary.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()

print("Reichsbank discount rate history:")
for era in ERA_ORDER:
    sub = ms[ms['era']==era]['reichsbank_discount_rate']
    print(f"  {era:15s}: avg={sub.mean():.1f}%, min={sub.min():.1f}%, max={sub.max():.1f}%")


---
## 6. 📉 Economic Collapse & Recovery — GDP, Unemployment, Trade

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

def shade_governments(ax):
    for s,e,col,label in [
        (1871,1918,GOLD,'Kaiserreich'),
        (1919,1932,GREEN,'Weimar Republic'),
        (1933,1944,RED,'Third Reich'),
        (1945,1948,BLUE,'Allied Occupation')]:
        ax.axvspan(s, e, alpha=0.06, color=col)

# Panel 1: GDP index
ax = axes[0,0]
shade_governments(ax)
ax.fill_between(econ['year'], econ['gdp_index_1913_100'], alpha=0.25, color=BLUE)
ax.plot(econ['year'], econ['gdp_index_1913_100'], color=BLUE, linewidth=2.5)
ax.axhline(100, color=GRAY, linewidth=0.8, linestyle='--', label='1913 baseline')
ax.set_title('German GDP Index (1913=100)', fontsize=11)
ax.set_ylabel('Index'); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
for yr, label in [(1918,'WWI end'),(1923,'Hyper'),(1929,'Wall St.'),(1933,'Hitler'),(1945,'WWII end')]:
    ax.axvline(yr, color=AMBER, linewidth=0.8, linestyle=':', alpha=0.7)
    ax.text(yr+0.2, 20, label, fontsize=7, color=AMBER, rotation=90)

# Panel 2: Unemployment
ax = axes[1,0]
shade_governments(ax)
ax.fill_between(econ['year'], econ['unemployment_rate_pct'],
                alpha=0.3, color=RED,
                where=econ['unemployment_rate_pct'] > 10)
ax.fill_between(econ['year'], econ['unemployment_rate_pct'],
                alpha=0.3, color=GREEN,
                where=econ['unemployment_rate_pct'] <= 5)
ax.plot(econ['year'], econ['unemployment_rate_pct'], color='#c9d1d9', linewidth=2)
ax.axhline(10, color=AMBER, linewidth=0.8, linestyle='--', label='10% threshold')
ax.set_title('Unemployment Rate (%)', fontsize=11)
ax.set_ylabel('%'); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
peak_u = econ['unemployment_rate_pct'].max()
peak_u_yr = econ.loc[econ['unemployment_rate_pct'].idxmax(), 'year']
ax.text(0.05, 0.88, f"Peak: {peak_u:.1f}% ({peak_u_yr})",
        transform=ax.transAxes, fontsize=9, color=RED,
        bbox=dict(boxstyle='round', facecolor='#21262d', alpha=0.8))

# Panel 3: Industrial production
ax = axes[0,1]
shade_governments(ax)
ax.fill_between(econ['year'], econ['industrial_production_idx'], alpha=0.2, color=GREEN)
ax.plot(econ['year'], econ['industrial_production_idx'], color=GREEN, linewidth=2)
ax.axhline(100, color=GRAY, linewidth=0.8, linestyle='--')
ax.set_title('Industrial Production Index (1913=100)', fontsize=11)
ax.set_ylabel('Index'); ax.grid(True, alpha=0.3)

# Panel 4: Budget balance
ax = axes[1,1]
shade_governments(ax)
ax.fill_between(econ['year'], econ['budget_balance_pct_gdp'], 0,
    where=econ['budget_balance_pct_gdp'] >= 0, alpha=0.4, color=GREEN, label='Surplus')
ax.fill_between(econ['year'], econ['budget_balance_pct_gdp'], 0,
    where=econ['budget_balance_pct_gdp'] < 0, alpha=0.4, color=RED, label='Deficit')
ax.plot(econ['year'], econ['budget_balance_pct_gdp'], color='#c9d1d9', linewidth=1.5)
ax.axhline(0, color=GRAY, linewidth=0.8)
ax.set_title('Government Budget Balance (% of GDP)', fontsize=11)
ax.set_ylabel('% GDP'); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.suptitle('German Economy 1871–1948: GDP, Unemployment, Industrial Production', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('economy.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()

print("GDP by era (avg index, 1913=100):")
for govt in ['Kaiserreich','Weimar Republic','Third Reich','Allied Occupation']:
    sub = econ[econ['government_in_power']==govt]['gdp_index_1913_100']
    print(f"  {govt:20s}: {sub.mean():.1f}")


---
## 7. 📰 Event Study — Geopolitical Shocks & Market Impact

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 7))

# Panel 1: FX shock by event type
ax = axes[0]
type_shock = events.groupby('event_type')['fx_rate_shock_pct'].mean().sort_values()
type_colors = {'monetary':BLUE,'war':RED,'political':AMBER,'financial':PURPLE,
               'diplomatic':GREEN,'economic':GRAY}
ax.barh(type_shock.index, type_shock.values,
        color=[type_colors.get(t,GRAY) for t in type_shock.index], alpha=0.85)
ax.axvline(0, color=GRAY, linewidth=0.8)
ax.set_title('Mean FX Shock by Event Type', fontsize=11)
ax.set_xlabel('Mean % change in Mark value'); ax.grid(True, alpha=0.3, axis='x')
for i,v in enumerate(type_shock.values):
    ax.text(v+(1 if v>=0 else -1), i, f'{v:+.0f}%', va='center', fontsize=8)

# Panel 2: Events on the log exchange rate chart
ax = axes[1]
ax.plot(fx['date_dt'], fx['log10_mark_per_usd'],
        color=GRAY, linewidth=1, alpha=0.7)
# Overlay events
for _, ev in events.iterrows():
    try:
        date_dt = pd.Timestamp(ev['date'] + '-01')
        sub = fx[fx['date_dt'] == date_dt]
        if sub.empty: continue
        log_rate = sub['log10_mark_per_usd'].iloc[0]
        col = RED if ev['fx_rate_shock_pct'] < 0 else GREEN
        ax.scatter([date_dt], [log_rate], color=col, s=60, zorder=5, alpha=0.8)
        if ev['severity'] in ['extreme','high']:
            ax.annotate(ev['event_name'][:20],
                        xy=(date_dt, log_rate),
                        xytext=(0, 12), textcoords='offset points',
                        fontsize=5.5, color=col, ha='center',
                        arrowprops=dict(arrowstyle='->', color=col, lw=0.5))
    except: pass
ax.set_title('Historical Events on Exchange Rate Timeline', fontsize=11)
ax.set_ylabel('log₁₀(Mark/USD)'); ax.grid(True, alpha=0.3)
red_p  = mpatches.Patch(color=RED,   label='Depreciation event')
green_p= mpatches.Patch(color=GREEN, label='Stabilization event')
ax.legend(handles=[red_p, green_p], fontsize=8)

# Panel 3: CPI shock vs FX shock scatter
ax = axes[2]
ev_plot = events[(events['fx_rate_shock_pct'].abs() < 10000) &
                 (events['cpi_shock_pct'].abs() < 10000)]
sc = ax.scatter(ev_plot['fx_rate_shock_pct'], ev_plot['cpi_shock_pct'],
                c=[{'extreme':3,'high':2,'medium':1,'low':0}[s]
                   for s in ev_plot['severity']],
                cmap='RdYlGn_r', s=80, alpha=0.8)
plt.colorbar(sc, ax=ax, label='Severity (0=low, 3=extreme)')
ax.axhline(0, color=GRAY, linewidth=0.7)
ax.axvline(0, color=GRAY, linewidth=0.7)
ax.set_title('FX Shock vs CPI Shock by Event', fontsize=11)
ax.set_xlabel('FX shock (%)'); ax.set_ylabel('CPI shock (%)')
ax.grid(True, alpha=0.3)
# Label a few
for _, ev in events[(events['severity']=='extreme') &
                    (events['fx_rate_shock_pct'].abs() < 5000)].iterrows():
    ax.annotate(ev['event_name'][:18],
                (ev['fx_rate_shock_pct'], ev['cpi_shock_pct']),
                fontsize=6, color='#c9d1d9',
                xytext=(5, 5), textcoords='offset points')

plt.suptitle('Historical Event Study — Market Impact Analysis', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('events.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()

print("Events by type and severity:")
print(events.groupby(['event_type','severity']).size().to_string())


---
## 8. 🔬 Quantity Theory of Money — Does MV = PQ Hold?

In [ ]:
# The Quantity Theory of Money: M × V = P × Q
# M = money supply, V = velocity, P = price level, Q = real output
# In log form: log(M) + log(V) = log(P) + log(Q)
# If V and Q are stable, then ΔM → ΔP (money printing causes inflation)

# Merge monthly data for the hyperinflation period
merged = fx[['date','date_dt','year','log10_mark_per_usd']].merge(
    inf[['date','log10_cpi','mom_inflation_pct']], on='date'
).merge(
    ms[['date','log10_money_supply','gold_coverage_pct']], on='date'
)

# Full period
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Panel 1: Log-log scatter: money supply vs price level
ax = axes[0,0]
sc = ax.scatter(merged['log10_money_supply'], merged['log10_cpi'],
                c=merged['year'], cmap='RdYlGn_r', s=10, alpha=0.7)
plt.colorbar(sc, ax=ax, label='Year')
# Regression line
slope, intercept, r, p, _ = stats.linregress(
    merged['log10_money_supply'].dropna(),
    merged['log10_cpi'].dropna())
x_line = np.linspace(merged['log10_money_supply'].min(),
                     merged['log10_money_supply'].max(), 100)
ax.plot(x_line, slope*x_line+intercept, color=RED, linewidth=2,
        label=f'OLS slope={slope:.2f}, r={r:.3f}')
ax.set_title('Money Supply vs CPI (log-log)', fontsize=11)
ax.set_xlabel('log₁₀(Money Supply)'); ax.set_ylabel('log₁₀(CPI)')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# Panel 2: Money supply growth rate vs inflation rate
ax = axes[0,1]
merged['ms_mom'] = merged['log10_money_supply'].diff()
ax.scatter(merged['ms_mom'].clip(-0.5,2),
           merged['mom_inflation_pct'].clip(-50, 5000),
           alpha=0.4, s=8, color=BLUE)
ax.set_title('Money Supply Growth vs Inflation (MoM)', fontsize=11)
ax.set_xlabel('Δlog₁₀ Money Supply'); ax.set_ylabel('MoM Inflation (%)')
ax.set_yscale('symlog', linthresh=10)
ax.grid(True, alpha=0.3)
r2,_ = stats.spearmanr(
    merged['ms_mom'].dropna(),
    merged['mom_inflation_pct'].dropna())
ax.text(0.05, 0.88, f'Spearman r = {r2:.3f}',
        transform=ax.transAxes, fontsize=9,
        bbox=dict(boxstyle='round', facecolor='#21262d', alpha=0.8))

# Panel 3: By era — how tight is the MV=PQ relationship?
ax = axes[1,0]
ERA_ORDER = ['Goldmark','Papiermark','Rentenmark','Reichsmark']
era_corrs = []
for era in ERA_ORDER:
    sub = merged[merged['year'].isin(
        fx[fx['era']==era]['year'].unique()
    )].dropna()
    if len(sub) > 10:
        r_era,_ = stats.pearsonr(sub['log10_money_supply'], sub['log10_cpi'])
        era_corrs.append((era, r_era))
if era_corrs:
    eras_e, corrs_e = zip(*era_corrs)
    ax.bar(eras_e, corrs_e,
           color=[ERA_COLORS[e] for e in eras_e], alpha=0.85)
    ax.axhline(0, color=GRAY, linewidth=0.8)
    ax.set_title('Pearson r: log(Money Supply) vs log(CPI) by Era', fontsize=11)
    ax.set_ylabel('Pearson r'); ax.set_ylim(-1, 1)
    ax.grid(True, alpha=0.3, axis='y')
    for i,(e,r_v) in enumerate(era_corrs):
        ax.text(i, r_v+0.03 if r_v>=0 else r_v-0.06, f'{r_v:.3f}',
                ha='center', fontsize=10)

# Panel 4: Velocity of money (implied: V = PQ/M)
ax = axes[1,1]
# V ∝ CPI / M (simplified, assuming Q roughly proportional to GDP)
merged['implied_velocity_log'] = merged['log10_cpi'] - merged['log10_money_supply']
for era, color in ERA_COLORS.items():
    sub = merged[merged['year'].isin(fx[fx['era']==era]['year'].unique())]
    ax.plot(sub['date_dt'], sub['implied_velocity_log'],
            color=color, linewidth=1.5, label=era, alpha=0.8)
ax.set_title('Implied Velocity of Money', fontsize=11)

ax.set_ylabel('Implied log(V)')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
ax.text(0.05, 0.88, 'Velocity rises sharply during hyperinflation',
        transform=ax.transAxes, fontsize=8,
        bbox=dict(boxstyle='round',facecolor='#21262d',alpha=0.8))
plt.savefig('quantity_theory.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()

print(f"Full period: log(M) vs log(CPI) Pearson r = {r:.3f}")
print(f"MoM growth vs inflation Spearman r = {r2:.3f}")





---
## 9. 📋 Key Findings

### Summary of the Four Eras

| Era | Period | Key Driver | FX stability |
|-----|--------|-----------|-------------|
| **Goldmark** | 1871–1914 | Gold standard | σ ≈ 0.06 (near-zero volatility) |
| **Papiermark** | 1914–1923 | War financing + reparations | 1 trillion× depreciation |
| **Rentenmark** | Nov 1923–1924 | Land-backed credibility | Overnight stabilization |
| **Reichsmark** | 1924–1948 | Controlled exchange rates | Stable officially; parallel market diverged under Nazi regime |

### Hyperinflation — Root Causes
The German hyperinflation of 1921–1923 was caused by three reinforcing factors:

1. **WWI war financing** — the Reichsbank printed money from 1914, destroying the gold standard. By November 1918, M had grown 4× with no increase in output.
2. **Reparations** — the Treaty of Versailles (1919) imposed 132 billion gold marks in reparations. Germany could not pay in gold; it printed Marks and bought foreign currency, accelerating the collapse.
3. **Ruhr occupation** (January 1923) — France seized Germany's industrial heartland. The government funded "passive resistance" by printing money. This was the trigger for terminal hyperinflation.

### The Rentenmark Miracle
On November 15, 1923, the Rentenmark was introduced at 1 Rentenmark = 10¹² Papiermark. Inflation stopped almost immediately. **Why?** Not because of gold — but because the government committed credibly to limiting supply to 3.2 billion Rentenmark. This is the first major historical demonstration that **credibility** matters as much as backing in monetary policy.

### Quantity Theory Confirmed
The log-log correlation between money supply and price level is r > 0.95 for the Papiermark era. The Quantity Theory of Money (MV = PQ) holds almost perfectly when velocity is non-constant — the "hot potato effect" (people spending money instantly to avoid holding depreciating currency) amplified the hyperinflation.

### The Great Depression in Germany
Germany's Great Depression was more severe than in the US or UK because:
- Germany could not devalue (Reichsmark was on gold exchange standard)
- Brüning government chose austerity (deflation) instead of stimulus
- Unemployment reached 30.1% by 1932
- This economic collapse directly contributed to Hitler's rise to power

---

*Dataset & notebook by **Sergey Nefedov** | [github.com/Sergpreneur](https://github.com/Sergpreneur)*  
*Sources: Bundesbank, Bresciani-Turroni (1937), NBER, Eichengreen (1992)*  
*If this helped your research, an upvote is greatly appreciated! 🙏*
